# Parsing de Documentos — Parte 2: HTML e Regex

**Problema:** os dados chegaram como HTML de um sistema web.
Têm tags, marcações, entidades HTML misturadas com o conteúdo útil.

**Ferramentas:**
- `BeautifulSoup` — navega a árvore de tags HTML
- `Regex` — pós-processa o texto extraído, encontra padrões

In [ ]:
!pip install beautifulsoup4 -q

In [ ]:
from bs4 import BeautifulSoup
import re

---
## Bloco 2a — BeautifulSoup: parsing de HTML

HTML tem **estrutura**: tags aninhadas formam uma árvore.
BeautifulSoup entende essa árvore e permite navegar por ela.

In [ ]:
# HTML bruto de um sistema de RH (exemplo realista)
html_bruto = """
<!DOCTYPE html>
<html>
<head>
    <title>Relatório de Funcionários - RH</title>
    <style>
        body { font-family: Arial; }
        .demitido { text-decoration: line-through; color: #999; }
        .destaque { background-color: #ffffcc; font-weight: bold; }
    </style>
</head>
<body>
    <h1>Relatório de Funcionários &mdash; Março/2026</h1>
    <p>Gerado em: 28/03/2026 às 14:30 | Responsável:
       <a href="mailto:maria.silva@empresa.com.br">maria.silva@empresa.com.br</a></p>

    <!-- Seção atualizada pelo jurídico em 25/03/2026 -->
    <h2>Equipe de Desenvolvimento</h2>
    <table border="1">
        <tr><th>Nome</th><th>CPF</th><th>Cargo</th><th>Salário</th><th>Status</th></tr>
        <tr>
            <td>Ana Paula Costa</td>
            <td>123.456.789-00</td>
            <td>Analista Sênior</td>
            <td>R$&nbsp;8.500,00</td>
            <td>Ativo</td>
        </tr>
        <tr>
            <td class="demitido">Bruno Martins</td>
            <td>987.654.321-00</td>
            <td>Desenvolvedor Jr</td>
            <td>R$&nbsp;4.200,00</td>
            <td><span class="demitido">~~Desligado em 15/02/2026~~</span></td>
        </tr>
        <tr>
            <td><span class="destaque">Carla Rodrigues</span></td>
            <td>456.789.123-00</td>
            <td>Gestora de Produto</td>
            <td>R$&nbsp;12.000,00</td>
            <td>Ativo — promovida</td>
        </tr>
    </table>

    <h2>Contatos</h2>
    <ul>
        <li>RH: rh@empresa.com.br | Tel: (62) 3333-4444</li>
        <li>TI: suporte@empresa.com.br</li>
        <li>Jurídico: juridico@empresa.com.br</li>
    </ul>
    <p><small>Sistema RH v3.2 — Documento interno, não distribuir.</small></p>
</body>
</html>
"""

print("=" * 60)
print("HTML BRUTO — o que chega do sistema de RH")
print("=" * 60)
print(html_bruto[:600])
print(f"\n... ({len(html_bruto)} caracteres no total)")

In [ ]:
# Parsing com BeautifulSoup
soup = BeautifulSoup(html_bruto, 'html.parser')

# 1. Texto limpo — sem nenhuma tag
print("=" * 60)
print("TEXTO LIMPO (soup.get_text)")
print("=" * 60)
texto_limpo = soup.get_text(separator='\n', strip=True)
print(texto_limpo)

In [ ]:
# 2. Navegar a árvore: encontrar elementos específicos
print("=" * 60)
print("NAVEGANDO A ÁRVORE HTML")
print("=" * 60)

titulo = soup.find('h1')
print(f"Título: {titulo.get_text()}")

print("\nSeções (h2):")
for h2 in soup.find_all('h2'):
    print(f"  → {h2.get_text()}")

print("\nLinks encontrados:")
for a in soup.find_all('a', href=True):
    print(f"  {a['href']} → {a.get_text()}")

print("\nItens de lista:")
for li in soup.find_all('li'):
    print(f"  • {li.get_text()}")

In [ ]:
# 3. Extrair tabela HTML → dados estruturados
print("=" * 60)
print("TABELA EXTRAÍDA COM BS4")
print("=" * 60)

tabela = soup.find('table')
dados = []
for linha in tabela.find_all('tr'):
    celulas = [td.get_text(strip=True) for td in linha.find_all(['th', 'td'])]
    dados.append(celulas)

for i, linha in enumerate(dados):
    if i == 0:
        print(' | '.join(f'{c:20s}' for c in linha))
        print('-' * 115)
    else:
        print(' | '.join(f'{c:20s}' for c in linha))

print(f"\n→ BS4 preservou a estrutura da tabela: {len(dados)-1} funcionários encontrados.")

---
## Bloco 2b — Regex como ferramenta transversal

Regex não é uma etapa isolada — ele aparece em **todo** pipeline
como ferramenta de limpeza e extração de padrões.

Vamos pós-processar o texto que o BS4 extraiu.

In [ ]:
# Extrair padrões do texto limpo com regex
texto = soup.get_text(separator='\n', strip=True)

print("=" * 60)
print("EXTRAINDO PADRÕES COM REGEX")
print("=" * 60)

# CPFs
cpfs = re.findall(r'\d{3}\.\d{3}\.\d{3}-\d{2}', texto)
print(f"\nCPFs encontrados:      {cpfs}")

# E-mails
emails = re.findall(r'[\w.+-]+@[\w-]+\.[\w.-]+', texto)
print(f"E-mails encontrados:   {emails}")

# Datas (dd/mm/aaaa)
datas = re.findall(r'\d{2}/\d{2}/\d{4}', texto)
print(f"Datas encontradas:     {datas}")

# Valores monetários
valores = re.findall(r'R\$\s?[\d.\xa0]+,\d{2}', texto)
print(f"Valores monetários:    {valores}")

# Telefones
telefones = re.findall(r'\(\d{2}\)\s?\d{4,5}-\d{4}', texto)
print(f"Telefones:             {telefones}")

In [ ]:
# Detectar marcadores de rasura e limpeza
print("=" * 60)
print("RASURAS E LIMPEZA")
print("=" * 60)

# Texto rasurado com marcadores ~~texto~~
rasuras = re.findall(r'~~(.+?)~~', texto)
print(f"\nTexto rasurado (~~texto~~): {rasuras}")

# Elementos com classe 'demitido' — BS4 + regex juntos
demitidos = soup.find_all(class_='demitido')
print("\nElementos marcados como 'demitido' (via BS4):")
for elem in demitidos:
    print(f"  ✗ {elem.get_text(strip=True)}")

# Limpeza: espaços múltiplos, quebras excessivas, entidades HTML residuais
texto_processado = re.sub(r'\n{3,}', '\n\n', texto)
texto_processado = re.sub(r' {2,}', ' ', texto_processado)
texto_processado = re.sub(r'\xa0', ' ', texto_processado)  # &nbsp;
print(f"\nTexto original: {len(texto)} chars → Limpo: {len(texto_processado)} chars")
print("\n--- TEXTO FINAL LIMPO ---")
print(texto_processado)

---
## Trade-offs

| Ferramenta | ✅ Vantagens | ❌ Limitações |
|---|---|---|
| **BeautifulSoup** | Entende a árvore HTML, navega por tags, preserva tabelas | Só funciona com HTML/XML |
| **Regex** | Rápido, sem dependências, controle total sobre padrões | Frágil com formatos variáveis, não entende semântica |

> **Na prática:** BS4 extrai o conteúdo, regex refina e encontra padrões.
> São ferramentas complementares.
>
> **Próximo desafio:** HTML está resolvido. Mas a maioria dos documentos
> corporativos está em **PDF**. BS4 não lê PDFs, regex não abre binários.